<a href="https://colab.research.google.com/github/johnyamarun/run-monitor/blob/main/Jack%20release%20v1.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

セル1：環境構築とGoogleドライブ連携

In [ ]:
# 1. 必要なライブラリのインストール
!pip install fitparse pandas google-generativeai
!pip install -U google-genai

# 2. Googleドライブのマウント
from google.colab import drive
import os

drive.mount('/content/drive')

# FITデータが格納されているディレクトリ
FIT_DIR = '/content/drive/MyDrive/FIT_Data/'
if not os.path.exists(FIT_DIR):
    print(f"ディレクトリが見つかりません: {FIT_DIR}")
    print("Googleドライブの直下に 'FIT_Data' フォルダを作成してください。")
else:
    print(f"ディレクトリを確認しました: {FIT_DIR}")

セル2：基礎データと期分けの設定（UIフォーム）

In [ ]:
# @title 🏃‍♂️ 基礎データ＆設定フォーム
# @markdown ---
# @markdown **🎯 レース目標設定（一般配布用）**
Target_Race_Name = "板橋Cityマラソン" # @param {type:"string"}
Target_Race_Date = "2026-03-15" # @param {type:"date"}
Target_Time_str = "2:33:00" # @param {type:"string"}

# @markdown ---
# @markdown **期分け（ピリオダイゼーション）**
Season = "マラソン期（秋冬）" # @param ["マラソン期（秋冬）", "スピード強化期（夏場）"]

# @markdown **基礎パラメーター**
Age = 39 # @param {type:"integer"}
Max_HR = 189 # @param {type:"integer"}
Rest_HR = 46 # @param {type:"integer"}

# @markdown **最新評価値（COROS等から）**
Current_VO2max = 67 # @param {type:"number"}
LT_HR = 158 # @param {type:"integer"}
LT_Pace_str = "3:29" # @param {type:"string"}
Target_Marathon_Pace_str = "3:37" # @param {type:"string"}
# @markdown ---

# --- 【Physiological Profile Bridge】 ---
RUNNER_MAX_HR = Max_HR
RUNNER_REST_HR = Rest_HR
RACE_NAME = Target_Race_Name
RACE_DATE = Target_Race_Date
RACE_TARGET_TIME = Target_Time_str

hrr = RUNNER_MAX_HR - RUNNER_REST_HR
print(f"✅ 設定を更新しました。")
print(f"🎯 目標レース: {RACE_NAME} ({RACE_DATE}) / 目標タイム: {RACE_TARGET_TIME}")
print(f"📊 最大心拍: {RUNNER_MAX_HR}bpm | 安静時心拍: {RUNNER_REST_HR}bpm | 予備心拍数(HRR): {hrr}bpm")

セル3：FITファイルの解析とデータ整形

In [ ]:
import glob
import os
import datetime
import pandas as pd
from fitparse import FitFile
import xml.etree.ElementTree as ET
from IPython.display import display

# FIT & TCX 両対応プレビュー
FIT_DIR = '/content/drive/MyDrive/FIT_Data/'

def get_preview_data(directory_path, max_files=50):
    files = glob.glob(os.path.join(directory_path, '*.fit')) + glob.glob(os.path.join(directory_path, '*.tcx'))
    files.sort(key=os.path.getmtime, reverse=True)
    sessions = []

    for file in files[:max_files]:
        try:
            hr, sp, v_osc, gct, cad = [], [], [], [], []
            start_time, total_dist = None, 0.0

            if file.lower().endswith('.fit'):
                fitfile = FitFile(file)
                for message in fitfile.get_messages('record'):
                    try:
                        data = message.get_values()
                        if not start_time and data.get('timestamp'): start_time = data['timestamp'] + datetime.timedelta(hours=9)
                        if data.get('heart_rate'): hr.append(data['heart_rate'])
                        if data.get('speed'): sp.append(data['speed'])
                        if data.get('distance'): total_dist = data['distance']
                        if data.get('stance_time'): gct.append(data['stance_time'])
                        if data.get('vertical_oscillation'): v_osc.append(data['vertical_oscillation'])
                        if data.get('cadence'): cad.append(data['cadence'])
                    except: continue

            elif file.lower().endswith('.tcx'):
                tree = ET.parse(file)
                root = tree.getroot()
                for elem in root.iter():
                    if '}' in elem.tag: elem.tag = elem.tag.split('}', 1)[1]
                times = root.findall('.//Time')
                if not times: continue
                start_time_str = times[0].text.replace('Z', '+00:00')
                start_time = datetime.datetime.fromisoformat(start_time_str) + datetime.timedelta(hours=9)
                distances = root.findall('.//DistanceMeters')
                total_dist = float(distances[-1].text) if distances else 0.0
                hrs = [int(h.text) for h in root.findall('.//HeartRateBpm/Value')]
                if hrs: hr = hrs
                if len(times) > 1 and total_dist > 0:
                    end_time_str = times[-1].text.replace('Z', '+00:00')
                    end_time = datetime.datetime.fromisoformat(end_time_str) + datetime.timedelta(hours=9)
                    seconds = (end_time - start_time).total_seconds()
                    avg_speed_ms = total_dist / seconds if seconds > 0 else 0
                    sp = [avg_speed_ms]

            if not start_time: continue

            avg_hr = sum(hr)/len(hr) if hr else 0
            max_hr_s = max(hr) if hr else 0
            avg_s = sum(sp)/len(sp) if sp else 0
            pace = f"{int((1000/avg_s)//60)}:{int((1000/avg_s)%60):02d}" if avg_s > 0 else "0:00"

            m_type = "Workout" if total_dist > 15000 or (max_hr_s) > 165 else "Jog"
            if total_dist < 3000: m_type = "Up/Down"

            sessions.append({
                "Date": start_time.strftime('%Y-%m-%d %H:%M'),
                "Menu": m_type,
                "Distance_km": round(total_dist/1000, 2),
                "Avg_Pace": pace,
                "HR_Avg_Max": f"{int(avg_hr)}/{int(max_hr_s)}",
                "Cadence": int(sum(cad)/len(cad))*2 if cad else 0,
                "GCT_ms": round(sum(gct)/len(gct), 1) if gct else 0.0,
                "VO_mm": round(sum(v_osc)/len(v_osc), 1) if v_osc else 0.0
            })
        except: continue

    return pd.DataFrame(sessions).sort_values('Date', ascending=False)

df_preview = get_preview_data(FIT_DIR)
display(df_preview)

In [ ]:
import google.generativeai as genai
from fitparse import FitFile
import pandas as pd
import datetime
import glob
import os
import xml.etree.ElementTree as ET
from google.colab import userdata, drive
from IPython.display import display, Markdown, HTML
import re
import html
import uuid

# --- 【Jack's Settings】 ---
Coach_Name = "Jack"

drive.mount('/content/drive', force_remount=True)
FIT_DIR = '/content/drive/MyDrive/FIT_Data/'

try:
    api_key = None
    for k in ['Gemini_API_Key', 'GEMINI_API_KEY']:
        try:
            api_key = userdata.get(k)
            if api_key: break
        except: continue
    if api_key:
        genai.configure(api_key=api_key)
    else:
        print("❌ APIキーが見つかりません。")
except Exception as e:
    print(f"❌ 認証エラー: {e}")

# Cell 2からのプロファイル＆レース目標読み込み
try:
    max_hr_limit = RUNNER_MAX_HR
    rest_hr_base = RUNNER_REST_HR
    race_name = RACE_NAME
    race_date = RACE_DATE
    race_target = RACE_TARGET_TIME
except NameError:
    max_hr_limit = 189
    rest_hr_base = 46
    race_name = "未設定レース"
    race_date = "2026-12-31"
    race_target = "未設定"

try:
    m_pace_str = Target_Marathon_Pace_str
    lt_pace_str = LT_Pace_str
except NameError:
    m_pace_str = "3:37"
    lt_pace_str = "3:29"

def time_to_sec(time_str):
    parts = str(time_str).split(':')
    if len(parts) >= 2: return int(parts[0]) * 60 + int(parts[1])
    return 0

m_pace_sec = time_to_sec(m_pace_str)
lt_pace_sec = time_to_sec(lt_pace_str)
hrr = max_hr_limit - rest_hr_base

def parse_all_data(directory_path, max_files=100):
    files = glob.glob(os.path.join(directory_path, '*.fit')) + glob.glob(os.path.join(directory_path, '*.tcx'))
    files.sort(key=os.path.getmtime, reverse=True)
    daily_logs = {}

    for file in files[:max_files]:
        try:
            hr, sp, v_osc, gct, cad = [], [], [], [], []
            sp_data = []
            start_time, total_dist = None, 0.0

            if file.lower().endswith('.fit'):
                fitfile = FitFile(file)
                for message in fitfile.get_messages('record'):
                    try:
                        data = message.get_values()
                        if not start_time and data.get('timestamp'): start_time = data['timestamp'] + datetime.timedelta(hours=9)

                        h = data.get('heart_rate')
                        s = data.get('speed')
                        g = data.get('stance_time')
                        v = data.get('vertical_oscillation')

                        if h is not None: hr.append(h)
                        if s is not None: sp.append(s)
                        if g is not None: gct.append(g)
                        if v is not None: v_osc.append(v)
                        if data.get('cadence'): cad.append(data['cadence'])
                        if data.get('distance'): total_dist = data['distance']

                        if s is not None or h is not None:
                            sp_data.append({'speed': s, 'hr': h, 'gct': g, 'vo': v})
                    except: continue

            elif file.lower().endswith('.tcx'):
                tree = ET.parse(file)
                root = tree.getroot()
                for elem in root.iter():
                    if '}' in elem.tag: elem.tag = elem.tag.split('}', 1)[1]
                times = root.findall('.//Time')
                if not times: continue
                start_time_str = times[0].text.replace('Z', '+00:00')
                start_time = datetime.datetime.fromisoformat(start_time_str) + datetime.timedelta(hours=9)
                distances = root.findall('.//DistanceMeters')
                total_dist = float(distances[-1].text) if distances else 0.0
                hrs = [int(h.text) for h in root.findall('.//HeartRateBpm/Value')]
                if hrs: hr = hrs
                if len(times) > 1 and total_dist > 0:
                    end_time_str = times[-1].text.replace('Z', '+00:00')
                    end_time = datetime.datetime.fromisoformat(end_time_str) + datetime.timedelta(hours=9)
                    seconds = (end_time - start_time).total_seconds()
                    avg_speed_ms = total_dist / seconds if seconds > 0 else 0
                    sp = [avg_speed_ms]

            if not start_time: continue
            date_key = start_time.strftime('%Y-%m-%d')

            avg_hr = sum(hr)/len(hr) if hr else 0
            max_hr_s = max(hr) if hr else 0
            avg_s = sum(sp)/len(sp) if sp else 0
            pace_sec_current = (1000/avg_s) if avg_s > 0 else 0
            pace = f"{int(pace_sec_current//60)}:{int(pace_sec_current%60):02d}" if pace_sec_current > 0 else "0:00"

            peak_pace_sec = 0
            peak_gct = 0.0
            gct_degradation = 0.0

            if sp_data and len(sp_data) > 20:
                valid_sp = [d for d in sp_data if d['speed'] is not None and d['speed'] > 0]
                if valid_sp:
                    valid_sp.sort(key=lambda x: x['speed'], reverse=True)
                    top_5_count = max(1, int(len(valid_sp) * 0.05))
                    top_data = valid_sp[:top_5_count]

                    peak_s = sum(d['speed'] for d in top_data) / top_5_count
                    peak_pace_sec = (1000/peak_s) if peak_s > 0 else 0

                    valid_gct = [d['gct'] for d in top_data if d['gct'] is not None]
                    if valid_gct: peak_gct = sum(valid_gct) / len(valid_gct)

                half_idx = len(sp_data) // 2
                fh_gct = [d['gct'] for d in sp_data[:half_idx] if d['gct'] is not None]
                sh_gct = [d['gct'] for d in sp_data[half_idx:] if d['gct'] is not None]
                if fh_gct and sh_gct:
                    gct_degradation = (sum(sh_gct)/len(sh_gct)) - (sum(fh_gct)/len(fh_gct))

            # --- 心拍ゾーン（Z3〜Z5）滞在時間の算出 ---
            z3_pts = z4_pts = z5_pts = 0
            if hr:
                for h in hr:
                    intensity = (h - rest_hr_base) / hrr if hrr > 0 else 0
                    if intensity >= 0.90: z5_pts += 1
                    elif intensity >= 0.80: z4_pts += 1
                    elif intensity >= 0.70: z3_pts += 1

            intensity_avg = (avg_hr - rest_hr_base) / hrr if hrr > 0 and avg_hr > 0 else 0
            intensity_max = (max_hr_s - rest_hr_base) / hrr if hrr > 0 and max_hr_s > 0 else 0

            # 生理学的判定（既存ロジック完全維持 ＋ Z3滞在によるLT1救済）
            if total_dist < 3000 and (intensity_avg < 0.60 or pace_sec_current > m_pace_sec + 60):
                m_type = "Up/Down"
            elif total_dist >= 20000:
                if (pace_sec_current > 0 and pace_sec_current <= m_pace_sec + 15) or intensity_avg >= 0.75:
                    m_type = "M-Pace LONG (特異的持久力/グリコーゲン枯渇)"
                else:
                    m_type = "Aerobic LONG (脂質代謝/有酸素ベース)"
            elif (peak_pace_sec > 0 and peak_pace_sec <= lt_pace_sec) or intensity_max >= 0.90:
                m_type = "VO2max / Interval (最大酸素摂取量/解糖系刺激)"
            elif (pace_sec_current > 0 and pace_sec_current <= m_pace_sec + 15) or intensity_avg >= 0.80 or intensity_max >= 0.85:
                m_type = "LT / M-Pace Tempo (乳酸閾値/特異的持久力)"
            elif total_dist < 3000:
                m_type = "Up/Down"
            else:
                # 【新規追加】リカバリー判定の前に、LT1領域（Z3以上）に10分以上滞在していればLT1走と判定
                if (z3_pts + z4_pts + z5_pts) > 600:
                    m_type = "Aerobic / LT1 (閾値下部/心肺刺激)"
                else:
                    m_type = "Recovery / E-Pace (アクティブリカバリー/毛細血管発達)"

            session = {
                "Time": start_time.strftime('%H:%M'), "Type": m_type, "Dist": round(total_dist/1000, 2),
                "Pace": pace, "PeakPaceSec": peak_pace_sec, "HR": f"{int(avg_hr)}/{int(max_hr_s)}",
                "Z3_min": z3_pts // 60, "Z4_min": z4_pts // 60, "Z5_min": z5_pts // 60,
                "Cadence": int(sum(cad)/len(cad))*2 if cad else 0,
                "GCT_Avg": round(sum(gct)/len(gct), 1) if gct else 0.0,
                "GCT_Peak": round(peak_gct, 1),
                "GCT_Deg": round(gct_degradation, 1),
                "VO_Avg": round(sum(v_osc)/len(v_osc), 1) if v_osc else 0.0
            }
            if date_key not in daily_logs: daily_logs[date_key] = []
            daily_logs[date_key].append(session)
        except Exception as e:
            continue

    sorted_logs = []
    weekdays = ['月', '火', '水', '木', '金', '土', '日']

    for d in sorted(daily_logs.keys(), reverse=True):
        sessions = daily_logs[d]
        sessions.sort(key=lambda x: x['Time'])

        d_obj = datetime.datetime.strptime(d, '%Y-%m-%d')
        date_str_with_weekday = f"{d_obj.strftime('%m/%d')} ({weekdays[d_obj.weekday()]})"

        daily_total_dist = sum(s['Dist'] for s in sessions)
        main_sessions = [s for s in sessions if s['Type'] != "Up/Down"]

        summary = ""
        if main_sessions:
            for s in main_sessions:
                gct_str = f", GCT(全体):{s['GCT_Avg']}ms" if s['GCT_Avg'] > 0 else ""
                vo_str = f", VO:{s['VO_Avg']}mm" if s['VO_Avg'] > 0 else ""

                peak_p = s.get('PeakPaceSec', 0)
                if peak_p > 0 and ('VO2max' in s['Type'] or 'Tempo' in s['Type'] or 'LT1' in s['Type']):
                    peak_pace_str = f"{int(peak_p//60)}:{int(peak_p%60):02d}/km"
                    gct_peak_str = f" (疾走時GCT: {s['GCT_Peak']}ms)" if s['GCT_Peak'] > 0 else ""
                    peak_str = f", 疾走推定:{peak_pace_str}{gct_peak_str}"
                else:
                    peak_str = ""

                deg_str = f", 後半GCT悪化:+{s['GCT_Deg']}ms" if s.get('GCT_Deg', 0) > 3.0 else ""

                # Z3〜Z5のゾーン滞在時間を追加してJackに伝達
                zone_total = s['Z3_min'] + s['Z4_min'] + s['Z5_min']
                zone_str = f", ゾーン滞在: Z3({s['Z3_min']}分) Z4({s['Z4_min']}分) Z5({s['Z5_min']}分)" if zone_total > 0 else ""

                summary += f"[{s['Type']}] {s['Dist']}km (全体Avg {s['Pace']}{peak_str}, HR:{s['HR']}{zone_str}{gct_str}{deg_str}{vo_str})\n"
        else:
            summary = "[Recovery] 調整\n"

        sorted_logs.append({
            "Date": date_str_with_weekday,
            "Daily_Total_km": round(daily_total_dist, 2),
            "Main_Menu": summary.strip()
        })
    return pd.DataFrame(sorted_logs)

# 実行とレース日数の計算
today = datetime.datetime.now().date()
try:
    race_date_obj = datetime.datetime.strptime(race_date, '%Y-%m-%d').date()
    days_to_race = (race_date_obj - today).days
except:
    days_to_race = "不明"

df = parse_all_data(FIT_DIR)
if not df.empty:
    weekly_dist = df['Daily_Total_km'].head(7).sum()

    latest_date = df.iloc[0]['Date']
    latest_menu = df.iloc[0]['Main_Menu'].replace('\n', ' / ')

    prompt = f"""
    お前は運動生理学の神髄を極めた全知全能のAI専属コーチ『{Coach_Name}』だ。

    【目標レース情報】
    - 大会名: {race_name}
    - レース日程: {race_date} (本番まであと {days_to_race} 日)
    - 目標タイム: {race_target}
    - 設定ペース: Mペース {m_pace_str}/km, LTペース {lt_pace_str}/km

    【絶対厳守のルール】
    - 素人のような感情表現は一切排除し、圧倒的な専門性を出せ。年齢は記載するな。
    - 本番までの残り日数（{days_to_race}日）を考慮し、テーパリングやピーキングの指示を具体的に行え。
    - 出力は必ず以下の3つのタグで囲んで分割しろ。それ以外の前置きは一切書くな。

    【提供データ】
    直近1週間の総走行距離: {weekly_dist} km
    全日程メニュー（ゾーン滞在時間、疾走時GCT、後半GCT悪化を考慮して分析しろ）:
    {df.to_string(index=False)}

    最新の練習（Xポスト用対象）: {latest_date} - {latest_menu}

    [THREADS_START]
    (ここにThreads用の投稿を作成。1週間の練習サマリーとJackの冷徹なレビューを合わせて「必ず500文字以内」に収めろ。絵文字適度に可。)
    [THREADS_END]

    [X_START]
    (ここにX用の投稿を作成。上記の「最新の練習」に対するJackの鋭いコメントと練習内容を合わせて「必ず140文字以内」で書け。)
    [X_END]

    [LAB_START]
    (ここにJ.Y.R. Labの内部解析レポートを作成。絵文字不可。論文レベルの極めて高度な運動生理学・バイオメカニクス用語を用い、負荷動態や疾走時の下肢剛性(GCT)の変化を物理的に深掘りしろ。マークダウン形式。)
    [LAB_END]
    """

    model = genai.GenerativeModel('gemini-2.5-flash')
    response = model.generate_content(prompt)
    res_text = response.text

    threads_match = re.search(r'\[THREADS_START\](.*?)\[THREADS_END\]', res_text, re.DOTALL)
    x_match = re.search(r'\[X_START\](.*?)\[X_END\]', res_text, re.DOTALL)
    lab_match = re.search(r'\[LAB_START\](.*?)\[LAB_END\]', res_text, re.DOTALL)

    threads_text = threads_match.group(1).strip() if threads_match else "取得エラー"
    x_text = x_match.group(1).strip() if x_match else "取得エラー"
    lab_text = lab_match.group(1).strip() if lab_match else "取得エラー"

    t_id = "t_" + str(uuid.uuid4()).replace("-", "")
    x_id = "x_" + str(uuid.uuid4()).replace("-", "")

    html_code = f"""
    <div style="display: flex; gap: 20px; margin-bottom: 20px;">
        <div style="flex: 1; background-color: #101010; padding: 15px; border-radius: 8px; border: 1px solid #333;">
            <h3 style="margin-top: 0; color: #ffffff;">🌀 Threads用（週次サマリー / 500字以内）</h3>
            <textarea id="{t_id}" style="width: 100%; height: 180px; background-color: #222; color: #fff; border: 1px solid #555; padding: 10px; border-radius: 4px; resize: vertical;">{html.escape(threads_text)}</textarea>
            <button onclick="navigator.clipboard.writeText(document.getElementById('{t_id}').value).then(()=>alert('Threads用テキストをコピーしました！'))" style="margin-top: 10px; padding: 8px 16px; background-color: #ffffff; color: #000000; font-weight: bold; border: none; border-radius: 4px; cursor: pointer;">📋 コピーする</button>
        </div>
        <div style="flex: 1; background-color: #f5f8fa; padding: 15px; border-radius: 8px; border: 1px solid #e1e8ed;">
            <h3 style="margin-top: 0; color: #1DA1F2;">🐦 X用（最新日のコメント / 140字以内）</h3>
            <textarea id="{x_id}" style="width: 100%; height: 180px; background-color: #fff; color: #14171a; border: 1px solid #ccd6dd; padding: 10px; border-radius: 4px; resize: vertical;">{html.escape(x_text)}</textarea>
            <button onclick="navigator.clipboard.writeText(document.getElementById('{x_id}').value).then(()=>alert('X用テキストをコピーしました！'))" style="margin-top: 10px; padding: 8px 16px; background-color: #1DA1F2; color: white; font-weight: bold; border: none; border-radius: 4px; cursor: pointer;">📋 コピーする</button>
        </div>
    </div>
    """
    display(HTML(html_code))
    display(Markdown("\n---\n### 🔬 J.Y.R. Lab: 内部解析レポート\n" + lab_text))

セル4：Gemini APIによる分析とレポート生成